# Data Merge Pipeline

Merges all 10 raw CSVs (FBref, FootballAPI, SportMonks) into a single team-level stats file.

**Output:** `processed/team_season_stats_merged.csv` — 12 rows, one per team, keyed by FootballAPI `team_id`.

In [ ]:
import pandas as pd
from pathlib import Path

RAW = Path("raw")
OUT = Path("processed")
OUT.mkdir(exist_ok=True)

## Load raw CSVs

In [ ]:
# FootballAPI
teams_fapi  = pd.read_csv(RAW / "footballapi/teams_footballapi.csv")
standings   = pd.read_csv(RAW / "footballapi/standings_footballapi.csv")
team_stats  = pd.read_csv(RAW / "footballapi/team_statistics_footballapi.csv")

# SportMonks
sm_stats    = pd.read_csv(RAW / "sportmonks/team_season_stats_sportmonks.csv")

# FBref
fbref_std   = pd.read_csv(RAW / "fbref/standard_fbref.csv")
fbref_keep  = pd.read_csv(RAW / "fbref/keeper_fbref.csv")
fbref_shoot = pd.read_csv(RAW / "fbref/shooting_fbref.csv")
fbref_misc  = pd.read_csv(RAW / "fbref/misc_fbref.csv")

print("Loaded all CSVs")

## Team Crosswalk

FootballAPI `team_id` is the canonical key. FBref only has squad names and SportMonks uses a different numeric ID system, so we map both to the FootballAPI ID via hard-coded dicts.

In [ ]:
FBREF_TO_FAPI_ID = {
    "Basel":         551,
    "FC Winterthur": 2180,
    "Grasshopper":   1013,
    "Lausanne-Sport": 1014,
    "Lugano":         606,
    "Luzern":         644,
    "Servette FC":   2184,
    "Sion":           630,
    "St. Gallen":    1011,
    "Thun":          1012,
    "Young Boys":     565,
    "Zürich":         783,
}

SM_TO_FAPI_ID = {
    13775: 606,   # Lugano
    10655: 1012,  # Thun
      362: 1011,  # St. Gallen
     3525: 2180,  # Winterthur
      468: 551,   # Basel
     3091: 644,   # Luzern
     2357: 1014,  # Lausanne Sport
      647: 565,   # Young Boys
     2866: 1013,  # Grasshopper
     1970: 2184,  # Servette
      462: 630,   # Sion
      389: 783,   # Zürich
}

# Crosswalk table for inspection
fbref_id_df = pd.DataFrame(FBREF_TO_FAPI_ID.items(), columns=["fbref_squad", "team_id"])
sm_id_df    = pd.DataFrame(SM_TO_FAPI_ID.items(), columns=["sm_team_id", "team_id"])

crosswalk = (
    teams_fapi[["team_id", "team_name"]]
    .merge(fbref_id_df, on="team_id", how="left")
    .merge(sm_id_df, on="team_id", how="left")
)
assert crosswalk[["fbref_squad", "sm_team_id"]].notna().all().all(), "Unmapped teams — check dicts above"
crosswalk

## Merge team-level stats

Start from the FootballAPI teams master as the base, then left-join each source in turn. All non-key columns are prefixed by source: `standings_`, `stats_`, `sm_`, `fbref_`.

In [ ]:
base = teams_fapi[["team_id", "team_name", "venue_name", "venue_city", "venue_capacity", "founded"]].copy()

# --- Standings ---
std_keep = [
    "rank", "points", "goals_diff", "form", "played", "win", "draw", "lose",
    "goals_for", "goals_against",
    "home_played", "home_win", "home_draw", "home_lose", "home_goals_for", "home_goals_against",
    "away_played", "away_win", "away_draw", "away_lose", "away_goals_for", "away_goals_against",
]
standings_sub = (
    standings[["team_id"] + std_keep]
    .rename(columns={c: f"standings_{c}" for c in std_keep})
)
merged = base.merge(standings_sub, on="team_id", how="left")

# --- FootballAPI team statistics ---
ts_cols = [c for c in team_stats.columns if c not in ("team_id", "team_name")]
team_stats_sub = (
    team_stats[["team_id"] + ts_cols]
    .rename(columns={c: f"stats_{c}" for c in ts_cols})
)
merged = merged.merge(team_stats_sub, on="team_id", how="left")

# --- SportMonks ---
sm_keep = [
    "offsides", "fouls", "ball_possession_pct", "injury_time_goals",
    "red_cards", "attacks", "yellow_cards", "rating",
    "yellowred_cards", "dangerous_attacks", "corners", "tackles", "shots",
]
sm_sub = sm_stats[["team_id"] + sm_keep].copy()
sm_sub["team_id"] = sm_sub["team_id"].map(SM_TO_FAPI_ID)
sm_sub = sm_sub.rename(columns={c: f"sm_{c}" for c in sm_keep})
merged = merged.merge(sm_sub, on="team_id", how="left")

# --- FBref standard (goals, assists, possession, age) ---
fbref_std["team_id"] = fbref_std["Squad"].map(FBREF_TO_FAPI_ID)
std_col_map = {
    "Poss":             "fbref_poss",
    "Age":              "fbref_age",
    "Playing Time_90s": "fbref_90s",
    "Performance_Gls":  "fbref_goals",
    "Performance_Ast":  "fbref_assists",
}
std_sub = fbref_std[["team_id"] + list(std_col_map)].rename(columns=std_col_map)
merged = merged.merge(std_sub, on="team_id", how="left")

# --- FBref keeper (saves, clean sheets, goals against) ---
fbref_keep["team_id"] = fbref_keep["Squad"].map(FBREF_TO_FAPI_ID)
keep_col_map = {
    "Performance_GA":    "fbref_goals_against",
    "Performance_GA90":  "fbref_goals_against_90",
    "Performance_Saves": "fbref_saves",
    "Performance_Save%": "fbref_save_pct",
    "Performance_CS":    "fbref_clean_sheets",
}
keep_sub = fbref_keep[["team_id"] + list(keep_col_map)].rename(columns=keep_col_map)
merged = merged.merge(keep_sub, on="team_id", how="left")

# --- FBref shooting ---
fbref_shoot["team_id"] = fbref_shoot["Squad"].map(FBREF_TO_FAPI_ID)
shoot_col_map = {
    "Standard_Sh":    "fbref_shots",
    "Standard_SoT":   "fbref_shots_on_target",
    "Standard_SoT%":  "fbref_shots_on_target_pct",
    "Standard_G/Sh":  "fbref_goals_per_shot",
}
shoot_sub = fbref_shoot[["team_id"] + list(shoot_col_map)].rename(columns=shoot_col_map)
merged = merged.merge(shoot_sub, on="team_id", how="left")

# --- FBref misc (discipline, defensive actions) ---
fbref_misc["team_id"] = fbref_misc["Squad"].map(FBREF_TO_FAPI_ID)
misc_col_map = {
    "Performance_CrdY": "fbref_yellow_cards",
    "Performance_CrdR": "fbref_red_cards",
    "Performance_Fls":  "fbref_fouls",
    "Performance_Off":  "fbref_offsides",
    "Performance_Crs":  "fbref_crosses",
    "Performance_Int":  "fbref_interceptions",
    "Performance_TklW": "fbref_tackles_won",
}
misc_sub = fbref_misc[["team_id"] + list(misc_col_map)].rename(columns=misc_col_map)
merged = merged.merge(misc_sub, on="team_id", how="left")

# Save
merged.to_csv(OUT / "team_season_stats_merged.csv", index=False)
print(f"Saved: {merged.shape[0]} rows x {merged.shape[1]} columns")
assert merged.shape[0] == 12, f"Expected 12 teams, got {merged.shape[0]}"
merged

## Sanity Checks

In [ ]:
print("=== Null counts per column ===")
nulls = merged.isnull().sum()
if nulls.any():
    print(nulls[nulls > 0].to_string())
else:
    print("No nulls")

print("\n=== Points vs wins (should correlate) ===")
print(
    merged[["team_name", "standings_points", "stats_fixtures_wins_total"]]
    .sort_values("standings_points", ascending=False)
    .to_string(index=False)
)

print("\n=== FC Thun spot-check ===")
thun = merged[merged["team_name"] == "FC Thun"].T
thun.columns = ["value"]
print(thun.to_string())